# CatBoost hyperparameter tuning

In [1]:
# Load data

import pandas as pd
import numpy as np 

train_df = pd.read_parquet("data/train_data.parquet")

train_df.index = pd.to_numeric(train_df.index, errors="coerce")

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop, errors="ignore")

In [2]:
# Separate features and targets

train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"]
train_full_y_reg = train_df["target_annual_roi"]

# Drop datetime features from the feature set

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_full_X = train_full_X.drop(columns=cols_to_drop, errors="ignore")

In [ ]:
# Create subsets of the data for different training sizes - chronplogical order is preserved, so we take the last N rows for each subset

train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)


cat_cols = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

## Classification

[Parameters](https://catboost.ai/docs/en/references/training-parameters/common)

### 1k

In [8]:
# Check the dates of 1k subset to ensure all data is from same month

print(train_1k_X["issue_d_month"].unique())
print(train_1k_X["issue_d_year"].unique())

<IntegerArray>
[10]
Length: 1, dtype: Int64
<IntegerArray>
[2016]
Length: 1, dtype: Int64


In [ ]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_cat.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_cat.head(200)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2.0, 10.0), # Class balancing weight for imbalanced target
        "objective": "Logloss", # Binary classification objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 3, 31) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostClassifier(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "scale_pos_weight",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_1k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_1k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_1k_parallel.html")


[I 2026-04-23 16:35:01,399] A new study created in memory with name: no-name-681235cd-8db3-4944-85f1-406d1b548fd2
[I 2026-04-23 16:35:05,173] Trial 0 finished with value: 0.6275497012565978 and parameters: {'iterations': 437, 'learning_rate': 0.1667521176194013, 'depth': 7, 'l2_leaf_reg': 0.24810409748678114, 'random_strength': 0.004207988669606638, 'rsm': 0.32479561626896214, 'border_count': 75, 'scale_pos_weight': 8.92940916619948, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bernoulli', 'score_function': 'L2', 'subsample': 0.3467236078827471}. Best is trial 0 with value: 0.6275497012565978.
[I 2026-04-23 16:35:07,718] Trial 1 finished with value: 0.6003255843773085 and parameters: {'iterations': 374, 'learning_rate': 0.0346466531747106, 'depth': 5, 'l2_leaf_reg': 0.014618962793704969, 'random_strength': 0.280163515871626, 'rsm': 0.3115950885216335, 'border_count': 119, 'scale_pos_weight': 4.930894746349534, 'boosting_type': 'Ordered', 'bootstrap_type': 'MVS', 'score_function': 'Co


BEST AUC: 0.6528
BEST PARAMETERS:
best_params = {
    "iterations": 128,
    "learning_rate": 0.0523043322856166,
    "depth": 4,
    "l2_leaf_reg": 0.1082138291061399,
    "random_strength": 4.268407710065499,
    "rsm": 0.39943378331909996,
    "border_count": 142,
    "scale_pos_weight": 8.04440910834439,
    "boosting_type": "Plain",
    "bootstrap_type": "MVS",
    "grow_policy": "Lossguide",
    "max_leaves": 26,
    "min_data_in_leaf": 31,
    "subsample": 0.9140471987919823,
}

--- PARAMETER IMPORTANCE ---
  random_strength     : 0.1965
  border_count        : 0.1528
  scale_pos_weight    : 0.1262
  rsm                 : 0.1127
  iterations          : 0.1110
  learning_rate       : 0.0814
  depth               : 0.0775
  boosting_type       : 0.0700
  l2_leaf_reg         : 0.0699
  bootstrap_type      : 0.0020


In [ ]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_catboost.best_params

best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "Logloss"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_catboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)


Optuna Cross-Val AUC: 0.6657
Holdout Test AUC:     0.6910


### 10k

In [ ]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_cat.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_cat.tail(2000)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2.0, 10.0), # Class balancing weight for imbalanced target
        "objective": "Logloss", # Binary classification objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 3, 31) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostClassifier(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "scale_pos_weight",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_10k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_10k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_10k_parallel.html")


[I 2026-04-23 16:37:10,923] A new study created in memory with name: no-name-376cd786-b63d-433e-a0a7-21f178ce57d6
[I 2026-04-23 16:37:16,857] Trial 0 finished with value: 0.6826721132382068 and parameters: {'iterations': 165, 'learning_rate': 0.03635998214912853, 'depth': 4, 'l2_leaf_reg': 0.004041328596876917, 'random_strength': 0.38118228380088937, 'rsm': 0.640104126686158, 'border_count': 161, 'scale_pos_weight': 5.4662804800947224, 'boosting_type': 'Plain', 'bootstrap_type': 'MVS', 'grow_policy': 'Depthwise', 'min_data_in_leaf': 98, 'score_function': 'L2', 'subsample': 0.3221214470047049}. Best is trial 0 with value: 0.6826721132382068.
[I 2026-04-23 16:37:23,839] Trial 6 finished with value: 0.6917901752839171 and parameters: {'iterations': 622, 'learning_rate': 0.014343211777190078, 'depth': 2, 'l2_leaf_reg': 0.028666358200160467, 'random_strength': 0.9671182788618011, 'rsm': 0.28942784636092417, 'border_count': 85, 'scale_pos_weight': 2.8349953491831004, 'boosting_type': 'Plain'


BEST AUC: 0.6986
BEST PARAMETERS:
best_params = {
    "iterations": 873,
    "learning_rate": 0.022633385345607094,
    "depth": 2,
    "l2_leaf_reg": 2.754771251713312,
    "random_strength": 0.42610255827673593,
    "rsm": 0.5575524745299817,
    "border_count": 148,
    "scale_pos_weight": 4.7286164041445975,
    "boosting_type": "Plain",
    "bootstrap_type": "Bayesian",
    "grow_policy": "SymmetricTree",
    "score_function": "L2",
    "bagging_temperature": 9.22990436697836,
}

--- PARAMETER IMPORTANCE ---
  depth               : 0.5852
  learning_rate       : 0.2399
  bootstrap_type      : 0.0653
  iterations          : 0.0467
  rsm                 : 0.0313
  l2_leaf_reg         : 0.0141
  random_strength     : 0.0069
  border_count        : 0.0053
  scale_pos_weight    : 0.0037
  boosting_type       : 0.0015


In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick


original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params["iterations"]}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "Logloss"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_catboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)


[Scaling Trick Applied] CATBOOST: Trees 873 -> 1746, LR 0.0226 -> 0.0113

Optuna Cross-Val AUC: 0.6986
Holdout Test AUC:     0.6725


### 100k

In [ ]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_cat.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_cat.tail(20000)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 10), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2.0, 10.0), # Class balancing weight for imbalanced target
        "objective": "Logloss", # Binary classification objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 5, 512) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostClassifier(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "scale_pos_weight",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_100k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_100k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_100k_parallel.html")


[I 2026-04-23 16:40:54,228] A new study created in memory with name: no-name-e2364fd3-e45b-4036-961b-dc1051975d65
[I 2026-04-23 16:41:31,352] Trial 2 finished with value: 0.7023057014509381 and parameters: {'iterations': 176, 'learning_rate': 0.032516407411224454, 'depth': 5, 'l2_leaf_reg': 4.674186697955896, 'random_strength': 0.006651114174251815, 'rsm': 0.42469100291849027, 'border_count': 179, 'scale_pos_weight': 9.67540188585603, 'boosting_type': 'Plain', 'bootstrap_type': 'Bernoulli', 'grow_policy': 'SymmetricTree', 'score_function': 'Cosine', 'subsample': 0.5080960756683879}. Best is trial 2 with value: 0.7023057014509381.
[I 2026-04-23 16:43:13,153] Trial 5 finished with value: 0.701920876662516 and parameters: {'iterations': 268, 'learning_rate': 0.11515247479277634, 'depth': 4, 'l2_leaf_reg': 0.024022768108595702, 'random_strength': 0.06017759634351341, 'rsm': 0.7967124872566056, 'border_count': 136, 'scale_pos_weight': 8.470071605662941, 'boosting_type': 'Plain', 'bootstrap_


BEST AUC: 0.7023
BEST PARAMETERS:
best_params = {
    "iterations": 176,
    "learning_rate": 0.032516407411224454,
    "depth": 5,
    "l2_leaf_reg": 4.674186697955896,
    "random_strength": 0.006651114174251815,
    "rsm": 0.42469100291849027,
    "border_count": 179,
    "scale_pos_weight": 9.67540188585603,
    "boosting_type": "Plain",
    "bootstrap_type": "Bernoulli",
    "grow_policy": "SymmetricTree",
    "score_function": "Cosine",
    "subsample": 0.5080960756683879,
}

--- PARAMETER IMPORTANCE ---
  scale_pos_weight    : 0.3541
  iterations          : 0.3009
  score_function      : 0.1645
  depth               : 0.0348
  random_strength     : 0.0328
  l2_leaf_reg         : 0.0281
  border_count        : 0.0278
  bootstrap_type      : 0.0215
  rsm                 : 0.0186
  boosting_type       : 0.0161
  learning_rate       : 0.0009


In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick
original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params["iterations"]}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "Logloss"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_catboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)


[Scaling Trick Applied] CATBOOST: Trees 176 -> 1760, LR 0.0325 -> 0.0033

Optuna Cross-Val AUC: 0.7023
Holdout Test AUC:     0.7146


### Full data set

In [ ]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 10000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 10), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 2.0, 10.0), # Class balancing weight for imbalanced target
        "objective": "Logloss", # Binary classification objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 5, 512) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostClassifier(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "scale_pos_weight",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_full_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_full_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Classification/optuna_catboost_full_parallel.html")


[I 2026-04-23 17:17:31,555] A new study created in memory with name: no-name-22cdb8b3-7ba1-4f27-85c5-a03cfe7c6f97


In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick


original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params["iterations"]}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "Logloss"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostClassifier(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Cross-Val AUC: {study_catboost.best_value:.4f}")
print(f"Holdout Test AUC:     {holdout_auc:.4f}")
print("="*40)

## Regression

[Parameters](https://catboost.ai/docs/en/references/training-parameters/common)

### 1k

In [ ]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_reg.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_reg.head(200)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "objective": "RMSE", # Regression objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 3, 31) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_1k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_1k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_1k_parallel.html")


[I 2026-04-23 21:01:43,515] A new study created in memory with name: no-name-10990126-c797-4ce7-9155-13a5a3264212
[I 2026-04-23 21:01:45,207] Trial 5 finished with value: 0.32538083333207507 and parameters: {'iterations': 188, 'learning_rate': 0.06210756615299539, 'depth': 6, 'l2_leaf_reg': 0.053333234561855056, 'random_strength': 1.4499367679162805, 'rsm': 0.572694989194898, 'border_count': 187, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 15, 'min_data_in_leaf': 90, 'bagging_temperature': 3.3034007713063493}. Best is trial 5 with value: 0.32538083333207507.
[I 2026-04-23 21:01:45,336] Trial 2 finished with value: 0.3464161621692088 and parameters: {'iterations': 481, 'learning_rate': 0.1456816868661623, 'depth': 2, 'l2_leaf_reg': 0.019742230570686765, 'random_strength': 0.5620822065927512, 'rsm': 0.36408439992054964, 'border_count': 174, 'boosting_type': 'Ordered', 'bootstrap_type': 'MVS', 'score_function': 'Cosine', 'subsample': 0


BEST RMSE: 0.3174
BEST PARAMETERS:
best_params = {
    "iterations": 403,
    "learning_rate": 0.010277333527010109,
    "depth": 5,
    "l2_leaf_reg": 0.0011087289969253184,
    "random_strength": 2.020311015063706,
    "rsm": 0.937390480286359,
    "border_count": 150,
    "boosting_type": "Plain",
    "bootstrap_type": "Bayesian",
    "grow_policy": "SymmetricTree",
    "score_function": "Cosine",
    "bagging_temperature": 2.769068135536773,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.8895
  bootstrap_type      : 0.0246
  rsm                 : 0.0205
  iterations          : 0.0181
  random_strength     : 0.0155
  border_count        : 0.0145
  boosting_type       : 0.0101
  depth               : 0.0043
  l2_leaf_reg         : 0.0028


In [ ]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_catboost.best_params

best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "RMSE"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_catboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)


Optuna Cross-Val RMSE: 0.3174
Holdout Test RMSE:     0.3327


### 10k

In [ ]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_reg.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_reg.tail(2000)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 8), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "objective": "RMSE", # Regression objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 3, 31) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 15, 100) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_10k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_10k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_10k_parallel.html")


[I 2026-04-23 21:05:00,109] A new study created in memory with name: no-name-c7248b73-e72e-4d8f-b258-4e3b026e409d
[I 2026-04-23 21:05:03,405] Trial 4 finished with value: 0.3015831765351672 and parameters: {'iterations': 288, 'learning_rate': 0.007757232182839749, 'depth': 2, 'l2_leaf_reg': 0.0071146714676037495, 'random_strength': 0.6944607590951303, 'rsm': 0.4850719859331596, 'border_count': 199, 'boosting_type': 'Plain', 'bootstrap_type': 'Bernoulli', 'grow_policy': 'Lossguide', 'max_leaves': 11, 'min_data_in_leaf': 43, 'subsample': 0.2934297647336404}. Best is trial 4 with value: 0.3015831765351672.
[I 2026-04-23 21:05:08,282] Trial 7 finished with value: 0.3077151583185739 and parameters: {'iterations': 876, 'learning_rate': 0.15328696065254524, 'depth': 2, 'l2_leaf_reg': 0.0032658694790414643, 'random_strength': 0.0267949273883958, 'rsm': 0.2717567590377965, 'border_count': 248, 'boosting_type': 'Plain', 'bootstrap_type': 'Bernoulli', 'grow_policy': 'Lossguide', 'max_leaves': 3, 


BEST RMSE: 0.2995
BEST PARAMETERS:
best_params = {
    "iterations": 875,
    "learning_rate": 0.008907815890416606,
    "depth": 8,
    "l2_leaf_reg": 0.024608726748244308,
    "random_strength": 3.9737936583221103,
    "rsm": 0.7631824789269197,
    "border_count": 231,
    "boosting_type": "Plain",
    "bootstrap_type": "MVS",
    "grow_policy": "Depthwise",
    "min_data_in_leaf": 51,
    "score_function": "Cosine",
    "subsample": 0.7027483673237973,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.4323
  random_strength     : 0.2010
  iterations          : 0.1586
  depth               : 0.1292
  rsm                 : 0.0365
  border_count        : 0.0232
  boosting_type       : 0.0096
  bootstrap_type      : 0.0090
  l2_leaf_reg         : 0.0007


In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick


original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params["iterations"]}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "RMSE"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_catboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)

SyntaxError: f-string: unmatched '[' (2773733064.py, line 15)

### 100k

In [ ]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning


X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_reg.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_reg.tail(20000)

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 1000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 10), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "objective": "RMSE", # Regression objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 5, 512) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=5) # TimeSeriesSplit to respect temporal order of data 5 splits for 5-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 5-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_100k_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_100k_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_100k_parallel.html")


[I 2026-04-23 16:40:54,228] A new study created in memory with name: no-name-e2364fd3-e45b-4036-961b-dc1051975d65
[I 2026-04-23 16:41:31,352] Trial 2 finished with value: 0.7023057014509381 and parameters: {'iterations': 176, 'learning_rate': 0.032516407411224454, 'depth': 5, 'l2_leaf_reg': 4.674186697955896, 'random_strength': 0.006651114174251815, 'rsm': 0.42469100291849027, 'border_count': 179, 'scale_pos_weight': 9.67540188585603, 'boosting_type': 'Plain', 'bootstrap_type': 'Bernoulli', 'grow_policy': 'SymmetricTree', 'score_function': 'Cosine', 'subsample': 0.5080960756683879}. Best is trial 2 with value: 0.7023057014509381.
[I 2026-04-23 16:43:13,153] Trial 5 finished with value: 0.701920876662516 and parameters: {'iterations': 268, 'learning_rate': 0.11515247479277634, 'depth': 4, 'l2_leaf_reg': 0.024022768108595702, 'random_strength': 0.06017759634351341, 'rsm': 0.7967124872566056, 'border_count': 136, 'scale_pos_weight': 8.470071605662941, 'boosting_type': 'Plain', 'bootstrap_


BEST AUC: 0.7023
BEST PARAMETERS:
best_params = {
    "iterations": 176,
    "learning_rate": 0.032516407411224454,
    "depth": 5,
    "l2_leaf_reg": 4.674186697955896,
    "random_strength": 0.006651114174251815,
    "rsm": 0.42469100291849027,
    "border_count": 179,
    "scale_pos_weight": 9.67540188585603,
    "boosting_type": "Plain",
    "bootstrap_type": "Bernoulli",
    "grow_policy": "SymmetricTree",
    "score_function": "Cosine",
    "subsample": 0.5080960756683879,
}

--- PARAMETER IMPORTANCE ---
  scale_pos_weight    : 0.3541
  iterations          : 0.3009
  score_function      : 0.1645
  depth               : 0.0348
  random_strength     : 0.0328
  l2_leaf_reg         : 0.0281
  border_count        : 0.0278
  bootstrap_type      : 0.0215
  rsm                 : 0.0186
  boosting_type       : 0.0161
  learning_rate       : 0.0009


In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick


original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params["iterations"]}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "RMSE"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_catboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)


[Scaling Trick Applied] CATBOOST: Trees 176 -> 1760, LR 0.0325 -> 0.0033

Optuna Cross-Val AUC: 0.7023
Holdout Test AUC:     0.7146


### Full data set

In [ ]:
import optuna
import numpy as np
import warnings
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler

# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_reg[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_reg[split_index:]

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 10000), # Number of boosting rounds (equivalent to n_estimators)
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "depth": trial.suggest_int("depth", 2, 10), # Max tree depth. Low to reduce overfitting on small datasets
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 100.0, log=True), # L2 regularization (equivalent to reg_lambda/reg_alpha)
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True), # Amount of randomness in scoring splits
        "rsm": trial.suggest_float("rsm", 0.2, 1.0), # Random subspace method (feature subsampling, equivalent to colsample_bytree)
        "border_count": trial.suggest_int("border_count", 64, 254), # Max bins for continuous features (equivalent to max_bin)
        "objective": "RMSE", # Regression objective
        "random_seed": 42, # Fixed seed for reproducibility
        "thread_count": 1, # Use single thread to avoid issues with parallelism in Optuna (equivalent to n_jobs)
        "verbose": False, # Suppress CatBoost output
        "boosting_type": trial.suggest_categorical("boosting_type", ["Plain", "Ordered"]), # Boosting type
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"]), # Bootstrap method for sampling data
        "cat_features": cat_cols, # Categorical features
        "verbose": False, # Suppress CatBoost output
    }

    if params["boosting_type"] == "Ordered": # Ordered boosting
        params["grow_policy"] = "SymmetricTree" # Growth of trees
    else:
        params["grow_policy"] = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]) # Growth of trees
  
    if params["grow_policy"] == "Lossguide": # Lossguide grow policy
        params["max_leaves"] = trial.suggest_int("max_leaves", 5, 512) # Max leaves per tree. Low to prevent memorizing small data
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = "L2" # Scoring function for split quality
    elif params["grow_policy"] == "Depthwise": # Depthwise grow policy
        params["min_data_in_leaf"] = trial.suggest_int("min_data_in_leaf", 10, 500) # Min data in one leaf (equivalent to min_child_samples)
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality
    else:
        params["score_function"] = trial.suggest_categorical("score_function", ["L2", "Cosine"]) # Scoring function for split quality

    if params["bootstrap_type"] == "Bayesian": # Bayesian bootstrap
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 10) # Intensity of Bayesian bootstrap
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.2, 1.0) # Subsample ratio
        
    
    tscv = TimeSeriesSplit(n_splits=3) # TimeSeriesSplit to respect temporal order of data 3 splits for 3-fold CV
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning): # 3-fold CV cycle
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        
        model = CatBoostRegressor(**params)
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_tr,
                y_tr
            )
        
        preds = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_catboost = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study_catboost.optimize(objective_catboost, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_catboost.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_catboost.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_catboost)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_catboost)
fig1.show()
    
fig2 = plot_param_importances(study_catboost)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_catboost, 
    params=[
        "iterations",
        "learning_rate",
        "depth",
        "l2_leaf_reg",
        "random_strength",
        "rsm",
        "border_count",
        "boosting_type",
        "bootstrap_type",
        "grow_policy"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_full_history.html")
fig2.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_full_importance.html")
fig3.write_html("hyperparameter_tuning/CatBoost/Regression/optuna_catboost_full_parallel.html")


[I 2026-04-23 17:17:31,555] A new study created in memory with name: no-name-22cdb8b3-7ba1-4f27-85c5-a03cfe7c6f97


In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_catboost.best_params.copy()

# Apply scaling trick


original_trees = best_params["iterations"]
original_lr = best_params["learning_rate"]

best_params["iterations"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] CATBOOST: Trees {original_trees} -> {best_params["iterations"]}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["thread_count"] = 7
best_params["verbose"] = False
best_params["objective"] = "RMSE"
best_params["cat_features"] = cat_cols
print(f"BEST PARAMS: {best_params}")

final_model = CatBoostRegressor(**best_params)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_model.fit(X_tuning, y_tuning, verbose=False)

holdout_preds = final_model.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Cross-Val RMSE: {study_catboost.best_value:.4f}")
print(f"Holdout Test RMSE:     {holdout_rmse:.4f}")
print("="*40)